In [1]:
import gymnasium as gym
from gymnasium import spaces
import pandas as pd
import random

class ConnectionsEnv(gym.Env):
    def __init__(self, csv_path="connections_training_dataset.csv"):
        super(ConnectionsEnv, self).__init__()
        
        # Load dataset and filter ONLY the real categories (label == 1) to build the board
        self.df = pd.read_csv(csv_path)
        if 'label' in self.df.columns:
            self.valid_groups_df = self.df[self.df['label'] == 1]
        else:
            self.valid_groups_df = self.df
            
        # The UI Controller: 16 buttons
        self.action_space = spaces.Discrete(16)
        # Dummy observation space since our Agents read the board directly
        self.observation_space = spaces.Discrete(1) 
        
        self.current_board = []
        self.solution_groups = []
        self.remaining_indices = []
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = [] 
        
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        
        # Build a valid 16-word board
        while True:
            sampled_rows = self.valid_groups_df.sample(4)
            self.solution_groups = []
            words_pool = []
            valid_board = True
            
            for _, row in sampled_rows.iterrows():
                group = {str(row['word1']).strip().lower(), 
                         str(row['word2']).strip().lower(), 
                         str(row['word3']).strip().lower(), 
                         str(row['word4']).strip().lower()}
                
                # Ensure no blank cells or duplicate words in a single row
                if len(group) != 4:
                    valid_board = False
                    break
                    
                self.solution_groups.append(group)
                words_pool.extend(list(group))
                
            # Ensure all 16 words on the board are entirely unique
            if valid_board and len(set(words_pool)) == 16:
                break
                
        # Shuffle the board
        random.shuffle(words_pool)
        self.current_board = words_pool
        self.remaining_indices = list(range(16))
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = []
        
        return 0, {}

    def load_custom_board(self, custom_groups):
        """
        Bypasses the random reset and loads a specific 16-word board.
        Expects a list of 4 lists, each containing 4 words.
        """
        self.solution_groups = []
        words_pool = []
        
        for group in custom_groups:
            # Clean and lowercase the inputs just to be safe
            cleaned_group = {str(w).strip().lower() for w in group}
            self.solution_groups.append(cleaned_group)
            words_pool.extend(list(cleaned_group))
            
        if len(set(words_pool)) != 16:
            print("WARNING: Your custom board does not have exactly 16 unique words!")
            
        # Shuffle the board so the AI doesn't just read them in order
        random.shuffle(words_pool)
        self.current_board = words_pool
        self.remaining_indices = list(range(16))
        self.lives = 4
        self.previous_guesses = set()
        self.current_selection = []
        
        return 0, {}
        
    def step(self, action):
        # 1. Invalid clicks (Removed words or already selected words)
        if action not in self.remaining_indices or action in self.current_selection:
            return 0, -1, False, False, {"status": "Invalid click"}
            
        # 2. Add to buffer
        self.current_selection.append(action)
        
        # 3. Waiting for 4 words
        if len(self.current_selection) < 4:
            return 0, 0, False, False, {"status": "Selecting"}
            
        # 4. We have 4 words! Evaluate the group.
        guess_indices = frozenset(self.current_selection)
        self.current_selection = []  # Clear buffer
        
        # Prevent immortal loop (guessing the exact same wrong thing twice)
        if guess_indices in self.previous_guesses:
            self.lives -= 1
            if self.lives <= 0:
                return 0, -15, True, False, {"status": "Game Over"}
            return 0, -10, False, False, {"status": "Repeated guess"}
            
        self.previous_guesses.add(guess_indices)
        guessed_words = {self.current_board[i] for i in guess_indices}
        
        terminated = False
        info = {}
        max_overlap = 0
        matched_group = None
        
        for group in self.solution_groups:
            overlap = len(guessed_words.intersection(group))
            if overlap > max_overlap:
                max_overlap = overlap
                matched_group = group

        # Feedback Logic
        if max_overlap == 4:
            # Remove the correct indices from the available pool
            self.remaining_indices = [i for i in self.remaining_indices if i not in guess_indices]
            self.solution_groups.remove(matched_group)
            info["status"] = "Correct!"
            
            if len(self.remaining_indices) == 0:
                terminated = True
                info["status"] = "Game Won!"
                
        elif max_overlap == 3:
            self.lives -= 1
            info["status"] = "One Away!"
            
        elif max_overlap == 2:
            self.lives -= 1
            info["status"] = "Two Away!"
            
        else:
            self.lives -= 1
            info["status"] = "Incorrect."
            
        if self.lives <= 0 and not terminated:
            terminated = True
            info["status"] = "Game Over"
            
        return 0, 0, terminated, False, info

In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import random
from sentence_transformers import SentenceTransformer
import numpy as np
import nltk
from nltk import pos_tag

# Ensure we have the NLTK grammar tagger
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

class DynamicFeatureExtractor:
    def __init__(self):
        print("Initializing HuggingFace MiniLM...")
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.vowels = set('aeiouy')
        self.pos_categories = ['NN', 'NNS', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'JJ', 'RB']
        print("Dynamic Extractor Ready!")

    def get_vector(self, word):
        """Takes ANY word and generates a dynamic 396-dimension vector."""
        word = str(word).strip().lower()
        
        # 1. MiniLM Subword Vector (384 dims)
        semantic_vec = self.model.encode(word)
        
        # 2. Word Length (1 dim)
        length_vec = np.array([len(word) / 15.0]) 
        
        # 3. Vowel Ratio (1 dim)
        num_vowels = sum(1 for char in word if char in self.vowels)
        vowel_ratio = np.array([num_vowels / max(1, len(word))])
        
        # 4. Part of Speech One-Hot (10 dims)
        tag = pos_tag([word])[0][1] 
        pos_vec = np.zeros(len(self.pos_categories))
        if tag in self.pos_categories:
            pos_vec[self.pos_categories.index(tag)] = 1.0
            
        # Glue them together: 384 + 1 + 1 + 10 = 396 Dimensions
        enriched_vec = np.concatenate([
            semantic_vec, 
            length_vec, 
            vowel_ratio, 
            pos_vec
        ]).astype(np.float32)
        
        return enriched_vec

extractor = DynamicFeatureExtractor()
class ConnectionsTripletDataset(Dataset):
    def __init__(self, csv_path, extractor):
        print("Loading dataset...")
        df = pd.read_csv(csv_path)
        
        # We ONLY care about the real, valid categories to train the gravitational pull
        if 'label' in df.columns:
            valid_df = df[df['label'] == 1].copy()
        else:
            valid_df = df.copy()
            
        self.groups = []
        words_set = set()
        
        # Extract the groups
        for _, row in valid_df.iterrows():
            group = [
                str(row['word1']).strip().lower(),
                str(row['word2']).strip().lower(),
                str(row['word3']).strip().lower(),
                str(row['word4']).strip().lower()
            ]
            if len(set(group)) == 4: # Ensure no weird blank duplicates
                self.groups.append(group)
                words_set.update(group)
                
        self.unique_words = list(words_set)
        
        # --- THE MASSIVE OPTIMIZATION ---
        print(f"Pre-computing 396-dim vectors for {len(self.unique_words)} unique words...")
        print("This will take a minute, but makes training 100x faster!")
        self.embeddings = {w: extractor.get_vector(w) for w in self.unique_words}
        print("Dataset Ready!")

    def __len__(self):
        # One epoch = looking at every valid group once
        return len(self.groups)

    def __getitem__(self, idx):
        group = self.groups[idx]
        
        # 1. Pick an Anchor and a Positive from the valid group
        anchor, positive = random.sample(group, 2)
        
        # 2. Pick a Negative (a word mathematically NOT in this group)
        negative = random.choice(self.unique_words)
        while negative in group:
            negative = random.choice(self.unique_words)
            
        # 3. Retrieve the pre-computed vectors
        anch_vec = torch.tensor(self.embeddings[anchor], dtype=torch.float32)
        pos_vec = torch.tensor(self.embeddings[positive], dtype=torch.float32)
        neg_vec = torch.tensor(self.embeddings[negative], dtype=torch.float32)
        
        return anch_vec, pos_vec, neg_vec

# --- INITIALIZE THE DATALOADER ---
# Assuming 'extractor' is already instantiated from earlier
# triplet_dataset = ConnectionsTripletDataset("connections_training_dataset.csv", extractor)

# # A batch size of 64 or 128 is usually the sweet spot for Triplet Loss
# train_loader = DataLoader(triplet_dataset, batch_size=64, shuffle=True)

Initializing HuggingFace MiniLM...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dynamic Extractor Ready!


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConnectionsSiameseNet(nn.Module):
    def __init__(self, input_dim=396, hidden_dim=256, output_dim=128):
        super(ConnectionsSiameseNet, self).__init__()
        
        # We take your 396-dim MiniLM + Engineered features and compress them
        # into a highly specialized 128-dim "NYT Connections" space.
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3), # increase when the validation isnt lowering with training
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward_one(self, x):
        # Passes a single word vector through the network
        x = self.encoder(x)
        # L2 Normalize the output so all vectors live on a unit sphere (crucial for clustering)
        return F.normalize(x, p=2, dim=1)

    def forward(self, anchor, positive, negative):
        # During training, we process all three simultaneously
        out_anchor = self.forward_one(anchor)
        out_pos = self.forward_one(positive)
        out_neg = self.forward_one(negative)
        return out_anchor, out_pos, out_neg

In [4]:
from sklearn.cluster import AgglomerativeClustering
import numpy as np

def solve_board_with_partitioning(sixteen_words, siamese_model, extractor):
    """
    Takes 16 words, projects them into the custom Siamese space, 
    and slices them into 4 groups.
    """
    siamese_model.eval()
    
    # 1. Extract base features using your existing MiniLM extractor
    base_vectors = [extractor.get_vector(w) for w in sixteen_words]
    base_tensor = torch.tensor(np.array(base_vectors), dtype=torch.float32)
    
    # 2. Warp the space using the trained Siamese Network
    with torch.no_grad():
        custom_embeddings = siamese_model.forward_one(base_tensor).numpy()
        
    # 3. The Partitioning Algorithm
    # We explicitly tell it to find exactly 4 clusters using Cosine distance
    clusterer = AgglomerativeClustering(
        n_clusters=4, 
        metric='cosine', 
        linkage='average'
    )
    
    # Predict the cluster labels (0, 1, 2, or 3) for each of the 16 words
    labels = clusterer.fit_predict(custom_embeddings)
    
    # 4. Group the final words
    final_groups = {0: [], 1: [], 2: [], 3: []}
    for word, label in zip(sixteen_words, labels):
        final_groups[label].append(word)
        
    return list(final_groups.values())


In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader

# 1. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")

# 2. Split the Dataset (80% Train, 20% Validation)
# Assuming 'triplet_dataset' is already initialized from the previous step
total_groups = len(triplet_dataset)
train_size = int(0.8 * total_groups)
val_size = total_groups - train_size

train_dataset, val_dataset = random_split(triplet_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
# We don't need to shuffle the validation set
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False) 

print(f"Training on {train_size} groups, Validating on {val_size} groups.")

# 3. Initialize Model, Loss, and Optimizer
model = ConnectionsSiameseNet(input_dim=396, hidden_dim=256, output_dim=128).to(device)
criterion = nn.TripletMarginLoss(margin=1.0, p=2)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. The Upgraded Training Loop
epochs = 100
print("\nCommencing Siamese Network Training with Validation...\n")

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    model.train() # Turn ON dropout and batch norm
    train_loss = 0.0
    
    for anchor, positive, negative in train_loader:
        anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
        
        optimizer.zero_grad()
        out_a, out_p, out_n = model(anchor, positive, negative)
        loss = criterion(out_a, out_p, out_n)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval() # Turn OFF dropout and batch norm for fair testing
    val_loss = 0.0
    
    with torch.no_grad(): # Disable gradient tracking to save memory/compute
        for anchor, positive, negative in val_loader:
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            
            out_a, out_p, out_n = model(anchor, positive, negative)
            loss = criterion(out_a, out_p, out_n)
            
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Print progress every epoch so you can watch the curves
    print(f"Epoch [{epoch+1:02d}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

print("\nTraining Complete")
torch.save(model.state_dict(), "nyt_siamese_weights.pth")
print("Model saved to 'nyt_siamese_weights.pth'")

Training on device: cpu
Training on 2861 groups, Validating on 716 groups.

Commencing Siamese Network Training with Validation...

Epoch [01/100] | Train Loss: 0.9350 | Val Loss: 0.8550
Epoch [02/100] | Train Loss: 0.8130 | Val Loss: 0.7965
Epoch [03/100] | Train Loss: 0.7692 | Val Loss: 0.7604
Epoch [04/100] | Train Loss: 0.7274 | Val Loss: 0.7229
Epoch [05/100] | Train Loss: 0.7308 | Val Loss: 0.7653
Epoch [06/100] | Train Loss: 0.7390 | Val Loss: 0.7491
Epoch [07/100] | Train Loss: 0.7245 | Val Loss: 0.7764
Epoch [08/100] | Train Loss: 0.7232 | Val Loss: 0.7534
Epoch [09/100] | Train Loss: 0.7126 | Val Loss: 0.7980
Epoch [10/100] | Train Loss: 0.6977 | Val Loss: 0.7590
Epoch [11/100] | Train Loss: 0.6914 | Val Loss: 0.7188
Epoch [12/100] | Train Loss: 0.6943 | Val Loss: 0.7324
Epoch [13/100] | Train Loss: 0.6777 | Val Loss: 0.7704
Epoch [14/100] | Train Loss: 0.6968 | Val Loss: 0.7357
Epoch [15/100] | Train Loss: 0.7056 | Val Loss: 0.7669
Epoch [16/100] | Train Loss: 0.6851 | Val L

In [ ]:
import torch
import numpy as np
import numpy as np
import torch
from itertools import combinations
from sklearn.metrics.pairwise import cosine_distances

class SiameseDetectiveAgent:
    def __init__(self, env, siamese_model, extractor):
        self.env = env
        self.model = siamese_model
        self.extractor = extractor
        self.constraints = []
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
    def score_combinations(self, remaining_words):
        """Evaluates the global board state to find the safest 4x4 partition."""
        combos = list(combinations(remaining_words, 4))
        valid_combos = []
        
        # 1. APPLY CONSTRAINTS (The Game Theory Pivot)
        for combo in combos:
            combo_set = set(combo)
            is_valid = True
            for constraint_type, target_set in self.constraints:
                overlap = len(combo_set.intersection(target_set))
                if constraint_type == "one_away" and overlap != 3:
                    is_valid = False
                    break
                elif constraint_type == "incorrect" and overlap >= 3:
                    is_valid = False
                    break
            if is_valid:
                valid_combos.append(combo)
                
        if not valid_combos:
            print("WARNING: Constraints are too tight, wiping memory and guessing blind!")
            self.constraints = []
            valid_combos = combos

        # 2. WARP THE SPACE
        base_vectors = [self.extractor.get_vector(w) for w in remaining_words]
        base_tensor = torch.tensor(np.array(base_vectors), dtype=torch.float32).to(self.device)
        
        with torch.no_grad():
            custom_embeddings = self.model.forward_one(base_tensor).cpu().numpy()
            
        dist_matrix = cosine_distances(custom_embeddings)
        word_to_idx = {w: i for i, w in enumerate(remaining_words)}
        
        # 3. SCORE THE LOCAL TIGHTNESS OF ALL COMBOS
        combo_scores = {}
        for combo in valid_combos:
            dist = 0
            indices = [word_to_idx[w] for w in combo]
            for i in range(4):
                for j in range(i+1, 4):
                    dist += dist_matrix[indices[i], indices[j]]
            combo_scores[combo] = dist / 6.0 # Average pairwise distance
            
        # Sort to find the locally tightest clusters
        ranked_local = sorted(combo_scores.keys(), key=lambda x: combo_scores[x])
        
        # 4. THE GLOBAL PARTITIONER (Beam Search Lookahead)
        final_rankings = []
        
        # We only check the "leftover board health" for the top 40 local guesses to save CPU
        candidates = ranked_local[:40] 
        
        for candidate in candidates:
            local_dist = combo_scores[candidate]
            leftovers = [w for w in remaining_words if w not in candidate]
            
            if len(leftovers) < 4:
                # If this guess clears the board, it is a perfect board state!
                global_dist = local_dist 
            else:
                # Calculate the best possible cluster hiding in the leftovers
                leftover_combos = list(combinations(leftovers, 4))
                best_leftover_dist = float('inf')
                
                for l_combo in leftover_combos:
                    dist = 0
                    l_indices = [word_to_idx[w] for w in l_combo]
                    for i in range(4):
                        for j in range(i+1, 4):
                            dist += dist_matrix[l_indices[i], l_indices[j]]
                    avg_dist = dist / 6.0
                    
                    if avg_dist < best_leftover_dist:
                        best_leftover_dist = avg_dist
                        
                # BLEND THE SCORES: 50% Current Guess + 50% Remaining Board Health
                # This explicitly punishes guesses that leave a messy board behind.
                global_dist = (local_dist * 0.5) + (best_leftover_dist * 0.5)
                
            final_rankings.append((candidate, global_dist))
            
        # Sort ASCENDING by the LOWEST global distance
        return sorted(final_rankings, key=lambda x: x[1])

    def play(self, custom_groups=None):
        if custom_groups:
            obs, info = self.env.load_custom_board(custom_groups)
        else:
            obs, info = self.env.reset()
            
        done = False
        turn_count = 1
        self.constraints = [] # Clean slate for a new game
        
        # print("="*50)
        # print("SIAMESE AI STARTING GAME")
        # print("="*50)
        
        while not done:
            remaining_words = [self.env.current_board[i] for i in self.env.remaining_indices]
            
            print(f"\n--- Turn {turn_count} (Lives: {self.env.lives}) ---")
            ranked_combos = self.score_combinations(remaining_words)
            
            # Pick the tightest cluster
            best_guess, best_dist = ranked_combos[0]
            print(f"Top Guess: {best_guess} (Cluster Distance: {best_dist:.3f})")
            
            guess_indices = [self.env.current_board.index(w) for w in best_guess]
            
            for idx in guess_indices:
                obs, reward, terminated, truncated, info = self.env.step(idx)
                
            status = info.get("status")
            print(f">> Result: {status}")
            
            # MEMORY AND CONSTRAINTS
            guess_set = set(best_guess)
            if status in ["Correct!", "Game Won!"]:
                #print(">> Pivot: Category solved! Purging obsolete constraints.")
                # Keep constraints ONLY if they don't include the words we just removed
                cleaned_constraints = []
                for c_type, c_set in self.constraints:
                    # If the constraint shares NO words with our correct guess, keep it!
                    if len(c_set.intersection(guess_set)) == 0:
                        cleaned_constraints.append((c_type, c_set))
                self.constraints = cleaned_constraints
                
            elif status == "One Away!":
                #print(">> Pivot: Activating Hard Constraint (Must share exactly 3 words)")
                self.constraints.append(("one_away", guess_set))
            else:
                #print(">> Pivot: Activating Soft Constraint (Avoiding this cluster)")
                self.constraints.append(("incorrect", guess_set))
                # Keep existing constraints for the rest of the board!
                pass 

            if terminated:
                done = True
                if status == "Game Won!":
                    print("\nTHE CLANKER BEAT THE GAME!")
                else:
                    print("\nTHE CLANKER RAN OUT OF LIVES.")
            
            turn_count += 1



In [6]:
import torch

# ==========================================
# 1. INITIALIZE THE "EYES" (Feature Extractor)
# ==========================================
extractor = DynamicFeatureExtractor() 

# ==========================================
# 2. INITIALIZE THE ENVIRONMENT
# ==========================================
csv_filename = "connections_training_dataset.csv" 
env = ConnectionsEnv(csv_path=csv_filename)

# ==========================================
# 3. INITIALIZE THE "BRAIN" (Siamese Network)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Loading Siamese Weights onto {device}...")

# Instantiate the model architecture
model = ConnectionsSiameseNet(input_dim=396, hidden_dim=256, output_dim=128).to(device)

# Load the weights 
try:
    model.load_state_dict(torch.load("nyt_siamese_weights.pth", map_location=device))
    model.eval() # Set to evaluation mode to disable Dropout
    print("Brain loaded successfully!")
except FileNotFoundError:
    print("WARNING: 'nyt_siamese_weights.pth' not found. Did you run the training loop?")

# ==========================================
# 4. INITIALIZE THE AGENT
# ==========================================
agent = SiameseDetectiveAgent(env, model, extractor)

# ==========================================
# 5. Run the Game
# ==========================================
# Let's test it against the board that previously broke the clustering algorithm
test_board = [
["Gut Feeling", "Intuition", "Hunch", "Sixth Sense"], 
["Do Not Disturb", "Ring", "Silent", "Vibrate"],
["Breadcrumb", "Catfish", "Ghost", "Love Bomb"],
["Air Cairo", "All Hallows", "Arm Warmer", "The Others"]
]

# (If you want to play a random board, just run: agent.play() instead)
agent.play(custom_groups=test_board)

Initializing HuggingFace MiniLM...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dynamic Extractor Ready!
Loading Siamese Weights onto cpu...
Brain loaded successfully!

THE CLANKER BEAT THE GAME!
